###J1 (1). Run EXPLAIN FORMATTED on gold.fact_sessions joined to gold.dim_charger,summing energy_kwh by connector_type . Name the join operator, why it was chosen,and the operator that marks a shuffle.


In [0]:
use catalog identifier(:catalog)

In [0]:

EXPLAIN FORMATTED
SELECT
  d.connector_type,
  SUM(f.energy_kwh) AS total_energy_kwh
FROM gold.fact_sessions f
JOIN gold.dim_charger d
  ON f.charger_id = d.charger_id
GROUP BY d.connector_type;



1. Join operator: look for SortMergeJoin (or BroadcastHashJoin).
* SortMergeJoin is chosen when both sides are large and neither side
fits comfortably in memory; it sorts both inputs on the join key and
merges them, which is the safest strategy for big-to-big joins.
* BroadcastHashJoin appears instead if the optimizer estimates one side
(usually dim_charger) is small enough to broadcast to every executor.
2. Shuffle marker: the Exchange operator. Every Exchange node = a shuffle
boundary (rows repartitioned/hashed on the join or grouping key).
HashAggregate / Sort under Exchange also hint at shuffle+sort stages.


###When is COPY INTO the better choice, and when Auto Loader? What does each use to remember loaded files?

**COPY INTO is better when:**

* You have a **small-to-medium dataset** (roughly < ~1 TB) and modest file volumes.
* Files arrive **infrequently or in predictable batches** (e.g., a nightly drop) — COPY INTO is a simple, idempotent SQL command you can schedule.


**Auto Loader is better when:**

* You have **large volumes of files** or a continuously growing directory (streaming source).
* You need **exactly-once, incremental ingestion** that scales to millions of files — file discovery stays cheap.


**How each remembers already-loaded files:**

* **COPY INTO** uses an **internal file-ingestion history/journal** stored alongside the target table — it records which files were loaded so re-running the same command skips them (idempotent). No setup required on your side.
* **Auto Loader** uses a **checkpoint directory** (`checkpointLocation`) backed by RocksDB state:
  * **Directory listing mode**: it lists the source directory and uses the checkpoint to identify which files are new.
  

**Rule of thumb:** COPY INTO for small, infrequent batch loads; Auto Loader for large-scale, continuous, or streaming ingestion.

###To promote the job to Prod (catalog vp_prod ) with Declarative Automation Bundles: name the config file, the section defining the job, how each environment gets its own catalog, and the deploy command.



**Declarative Automation Bundles (DABs) — promoting the job to Prod (catalog `vp_prod`)**

**1. Config file:** `databricks.yml` (the bundle's main configuration file at the project root — can be overridden with `-f/--file`). It contains the bundle name, default target, and the resource, variable, and workspace definitions.

**2. Section defining the job:** the `resources.jobs` section. Example:

```yaml
resources:
  jobs:
    my_elt_job:
      name: elt-job-${bundle.target}
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone: "UTC"
      tasks:
        - task_key: run_pipeline
          pipeline_task:
            pipeline_id: ${resources.pipelines.my_elt_pipeline.id}
```

**3. How each environment gets its own catalog:** via **targets** (one per environment, e.g. `dev` and `prod`) combined with **variable substitution** (`${var.catalog}`). You define the variable per target so the same bundle deploys against a different catalog in each environment:

```yaml
variables:
  catalog:
    description: UC catalog for this environment

targets:
  dev:
    variables:
      catalog: vp_dev
  prod:
    variables:
      catalog: vp_prod
```

At runtime the pipeline/notebook code uses `use catalog identifier(:catalog)` (as in this notebook) and DABs pass the target-appropriate `catalog` variable — the dev deploy writes to `vp_dev` while the prod deploy writes to `vp_prod`, with no code changes.

**4. Deploy command:**

```bash
databricks bundle deploy --target prod
```

(Verify first with `databricks bundle validate --target prod`; run it with `databricks bundle run --target prod`.)
